# Wishlist Click 로그 SQL 생성
- products.csv에서 productId, productName, productCategory 랜덤 샘플링
- productCategory: VALID_CATEGORIES 8개로 pool 필터링
- actionType: add / remove 랜덤
- user_id: 1~100 (로그인) / null (비로그인)
- user_login_id: user0001~user0100 (로그인) / null (비로그인)
- client_uuid: 세션마다 고유 UUID
- CSV 파일은 이 노트북과 같은 디렉토리에 위치해야 함

In [ ]:
import random
import json
import uuid
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
START_DATE      = datetime(2025, 6, 1, 0, 0, 0)
END_DATE        = datetime(2026, 6, 16, 23, 59, 59)
ROW_COUNT       = 500   # 생성할 로그 수

# 비로그인 사용자 비율 (0.0 ~ 1.0)
ANONYMOUS_RATIO = 0.3

# 허용 카테고리
VALID_CATEGORIES = ['디지털/가전', '패션의류', '패션잡화', '화장품/미용', '식품', '생활/건강', '스포츠/레저', '가구/인테리어']

In [ ]:
# products.csv 로드
products_df = pd.read_csv('products.csv')

# is_active=1 필터 + VALID_CATEGORIES 필터 + 필요한 컬럼만 추출
products_df = products_df[
    (products_df['is_active'] == 1) &
    (products_df['product_category'].isin(VALID_CATEGORIES))
][['product_id', 'name', 'product_category']].dropna(subset=['product_id', 'name'])

product_pool = products_df.to_dict('records')

print(f'✅ products.csv 로드 완료 → 상품 {len(product_pool)}개')
print(products_df.head(3))

In [ ]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

In [ ]:
rows = []

for _ in range(ROW_COUNT):
    # event_timestamp 기준으로 먼저 뽑고, history_timestamp = event_ts + 1초 (서버 수신이 더 늦음)
    event_ts   = random_datetime(START_DATE, END_DATE)
    history_ts = event_ts + timedelta(seconds=1)

    # products.csv에서 랜덤 상품 선택
    product          = random.choice(product_pool)
    product_id       = str(product['product_id'])
    product_name     = product['name']
    product_category = product['product_category']

    # actionType: add / remove 랜덤
    action_type = random.choice(['add', 'remove'])

    client_uuid = str(uuid.uuid4())

    # 비로그인 사용자 여부
    is_anonymous = random.random() < ANONYMOUS_RATIO

    if is_anonymous:
        user_id       = None
        user_login_id = None
    else:
        user_id       = random.randint(1, 100)
        user_login_id = f'user{user_id:04d}'

    json_log = json.dumps({
        'event_name':      'wishlist_click',
        'productName':     product_name,
        'productId':       product_id,
        'productCategory': product_category,
        'actionType':      action_type,
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(event_ts)
    }, ensure_ascii=False)

    rows.append((history_ts, json_log))

print(f'✅ {ROW_COUNT}개 wishlist_click 로그 생성 완료')

In [ ]:
# SQL 생성 및 저장
lines  = ['INSERT INTO first_save_history (history_timestamp, json_log) VALUES']
values = []

for history_ts, json_log in rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('wishlist_click_logs.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {ROW_COUNT}개 wishlist_click 로그 SQL 생성 완료 → wishlist_click_logs.sql')

In [ ]:
# ── 미리보기 ──
print('=== WISHLIST CLICK SQL (앞 500자) ===')
print(sql[:500])